## **Análisis EDA: Kardex Enero 2026**

In [1]:
import pandas as pd
import numpy as np
import math

In [2]:
# Importar base de datos
data = pd.read_excel('../data/raw/kardex_202607.xlsx')

# Asegurar tipo de datos de fecha
data['PERIODO'] = pd.to_datetime(data['PERIODO'], errors='coerce')
data['FULCOM'] = pd.to_datetime(data['FULCOM'], errors='coerce')

# Cambiar el periodo al final del mes (01 -> 28 / 30 / 31)
data['PERIODO'] = data['PERIODO'] + pd.offsets.MonthEnd(0)

# Asegurar datos numéricos
columnas_numericas = ['STOCK', 'COSTO', 'VALORIZACION', 'UNID CONV']

for col in columnas_numericas:
    data[col] = pd.to_numeric(data[col], errors='coerce')

data.head()

,PERIODO,CODIGO,DESCRIPTOR,STOCK,COSTO,VALORIZACION,FULCOM,UNID CONV,LINEA,NOMBRE LINEA,POLITICA CJE AL 30SEP
0,2026-07-31,SKU_000001,PRODUCTO_000001,354,9404.62,3329235.48,2026-07-23,1.0,400,FARMACIA,P1
1,2026-07-31,SKU_000002,PRODUCTO_000002,464,10545.13,4892940.32,2026-10-22,1.0,400,FARMACIA,P0
2,2026-07-31,SKU_000003,PRODUCTO_000003,39,10294.11,401470.29,2026-05-25,1.0,400,FARMACIA,P1
3,2026-07-31,SKU_000004,PRODUCTO_000004,568,17602.22,9998060.96,2026-10-22,1.0,400,FARMACIA,P1
4,2026-07-31,SKU_016589,PRODUCTO_016642,1,4614.00,4614.00,2025-08-24,1.0,400,FARMACIA,P0


In [3]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 15357 entries, 0 to 15356
Data columns (total 11 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   PERIODO                15357 non-null  datetime64[us]
 1   CODIGO                 15357 non-null  str           
 2   DESCRIPTOR             15357 non-null  str           
 3   STOCK                  15357 non-null  int64         
 4   COSTO                  15357 non-null  float64       
 5   VALORIZACION           15357 non-null  float64       
 6   FULCOM                 15357 non-null  datetime64[us]
 7   UNID CONV              15355 non-null  float64       
 8   LINEA                  15357 non-null  int64         
 9   NOMBRE LINEA           15357 non-null  str           
 10  POLITICA CJE AL 30SEP  15357 non-null  str           
dtypes: datetime64[us](2), float64(3), int64(2), str(4)
memory usage: 1.3 MB


In [4]:
# 1. MESES ANTIG.
data["MESES ANTIG."] = np.where(data["FULCOM"].isna(), np.nan, np.maximum(0, ((data["PERIODO"] + pd.offsets.MonthEnd(0)).dt.year - data["FULCOM"].dt.year) * 12 
                                                                          + (data["PERIODO"] + pd.offsets.MonthEnd(0)).dt.month - data["FULCOM"].dt.month))

# 2. ALERTA ANTIG.
data["ALERTA ANTIG."] = np.select([data["FULCOM"].isna(), 
                                   data["MESES ANTIG."] > 12, 
                                   data["MESES ANTIG."] > 6], ["SIN FECHA", "ROJO", "ÁMBAR"], default="VERDE")

# 3. DIF. VALORIZACIÓN
data["DIF. VALORIZACIÓN"] = data["VALORIZACION"] - (data["STOCK"] * data["COSTO"])

# 4. ALERTA STOCK
data["ALERTA STOCK"] = np.select([data["STOCK"] < 0, data["STOCK"] == 0], ["NEGATIVO", "CERO"], default="OK")

# 5. ALERTA COSTO
data["ALERTA COSTO"] = np.where(data["COSTO"] <= 0, "INVÁLIDO", "OK")

# 6. ALERTA CONVERSIÓN
data["ALERTA CONVERSIÓN"] = np.select([data["UNID CONV"].isna() | (data["UNID CONV"] <= 0), data["UNID CONV"] != 1], ["INVÁLIDA", "REVISAR"], default="OK")

# 7. ESTADO GENERAL
data["ESTADO GENERAL"] = np.select([(data["ALERTA ANTIG."] == "ROJO") | 
                                    (data["ALERTA STOCK"] != "OK") | 
                                    (data["ALERTA COSTO"] != "OK") | 
                                    (data["DIF. VALORIZACIÓN"].abs() > 1) | 
                                    (data["ALERTA CONVERSIÓN"] != "OK"), data["ALERTA ANTIG."] == "ÁMBAR"], ["REVISAR", "MONITOREAR"], default="OK")

In [5]:
data

,PERIODO,CODIGO,DESCRIPTOR,STOCK,COSTO,VALORIZACION,FULCOM,UNID CONV,LINEA,NOMBRE LINEA,POLITICA CJE AL 30SEP,MESES ANTIG.,ALERTA ANTIG.,DIF. VALORIZACIÓN,ALERTA STOCK,ALERTA COSTO,ALERTA CONVERSIÓN,ESTADO GENERAL
0,2026-07-31,SKU_000001,PRODUCTO_000001,354,9404.62,3329235.48,2026-07-23,1.0,400,FARMACIA,P1,0.0,VERDE,0.0,OK,OK,OK,OK
1,2026-07-31,SKU_000002,PRODUCTO_000002,464,10545.13,4892940.32,2026-10-22,1.0,400,FARMACIA,P0,0.0,VERDE,0.0,OK,OK,OK,OK
2,2026-07-31,SKU_000003,PRODUCTO_000003,39,10294.11,401470.29,2026-05-25,1.0,400,FARMACIA,P1,2.0,VERDE,0.0,OK,OK,OK,OK
3,2026-07-31,SKU_000004,PRODUCTO_000004,568,17602.22,9998060.96,2026-10-22,1.0,400,FARMACIA,P1,0.0,VERDE,0.0,OK,OK,OK,OK
4,2026-07-31,SKU_016589,PRODUCTO_016642,1,4614.00,4614.00,2025-08-24,1.0,400,FARMACIA,P0,11.0,ÁMBAR,0.0,OK,OK,OK,MONITOREAR
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15352,2026-07-31,SKU_015268,PRODUCTO_015254,350,29.15,10202.50,2025-07-23,1.0,400,FARMACIA,P0,12.0,ÁMBAR,0.0,OK,OK,OK,MONITOREAR
15353,2026-07-31,SKU_015269,PRODUCTO_015255,1,21.01,21.01,2023-07-03,1.0,400,FARMACIA,P0,36.0,ROJO,0.0,OK,OK,OK,REVISAR
15354,2026-07-31,SKU_015270,PRODUCTO_015256,1,17.65,17.65,2023-07-03,1.0,452,BELLEZA,P0,36.0,ROJO,0.0,OK,OK,OK,REVISAR
15355,2026-07-31,SKU_015271,PRODUCTO_015257,1,5297.75,5297.75,2025-08-22,1.0,999,DISTRIBUIBLES,P0,11.0,ÁMBAR,0.0,OK,OK,OK,MONITOREAR


## Cálculos Relevantes para el Criterio de Alertas

In [6]:
# Reglas de conciliación
saldo_gl_cta_000184 = 117495620727  # Dato recuperado de los archivos originales
total_jul = data['VALORIZACION'].sum()
recetario_jul = 520502056 # Dato recuperado de los archivos originales
gran_total_jul = total_jul + recetario_jul

if (abs(saldo_gl_cta_000184 - gran_total_jul) <= saldo_gl_cta_000184 * 0.001):
    conciliacion = "CONCILIADO"
else:
    conciliacion = "NO CONCILIADO"

# Reglas de antiguedad
rojos = (data['ALERTA ANTIG.'] == 'ROJO').sum()
ambar = (data['ALERTA ANTIG.'] == 'ÁMBAR').sum()
rojos_clp = data[data['ALERTA ANTIG.'] == 'ROJO']['VALORIZACION'].sum()
ambar_clp = data[data['ALERTA ANTIG.'] == 'ÁMBAR']['VALORIZACION'].sum()

if (((data['ALERTA ANTIG.'] == 'ROJO').sum() != 0) | 
    ((data['ALERTA ANTIG.'] == 'ÁMBAR').sum() != 0)):
    antiguedad_eval = "NO ALINEADO"
else:
    antiguedad_eval = "ALINEADO"

# Reglas de fechas futuras FULCOM
fechas_futuras = (data['FULCOM'] > data['PERIODO']).sum()
fechas_futuras_clp = data[data['FULCOM'] > data['PERIODO']]['VALORIZACION'].sum()

if fechas_futuras > 0:
    fechas_futuras_eval = "CRÍTICO"
else:
    fechas_futuras_eval = "NO CRÍTICO"

# Reglas de conversión
conversiones_mayores = (data['ALERTA CONVERSIÓN'] == 'REVISAR').sum()
conversiones_nulas = (data['UNID CONV'].isna()).sum()
conversiones_mayores_clp = data[data['ALERTA CONVERSIÓN'] == 'REVISAR']['VALORIZACION'].sum()
conversiones_nulas_clp = data[data['UNID CONV'].isna()]['VALORIZACION'].sum()

if (conversiones_mayores > 0 | conversiones_nulas > 0):
    conversiones_eval = "REVISAR"
else:
    conversiones_mayores_eval = "OK"

# Reglas de valorización
valorizacion_inconsistente = (round(data['DIF. VALORIZACIÓN']) > 0).sum()
valorizacion_clp = data[round(data['DIF. VALORIZACIÓN']) > 0]['VALORIZACION'].sum()

if valorizacion_inconsistente > 0:
    valorizacion_eval = "REVISAR"
else:
    valorizacion_eval = "OK"

# Reglas de duplicados
codigos_duplicados = (data['CODIGO'].duplicated()).sum()
duplicados_clp = data[data['CODIGO'].duplicated()]['VALORIZACION'].sum()

if codigos_duplicados > 0:
    duplicados_eval = "REVISAR"
else:
    duplicados_eval = "OK"

# Reglas de política sin canje (P0) envejecido
p0_rojo = ((data['ALERTA ANTIG.'] == 'ROJO') & (data['POLITICA CJE AL 30SEP'] == 'P0')).sum()
p0_ambar = ((data['ALERTA ANTIG.'] == 'ÁMBAR') & (data['POLITICA CJE AL 30SEP'] == 'P0')).sum()
p0_rojo_clp = data[(data['ALERTA ANTIG.'] == 'ROJO') & (data['POLITICA CJE AL 30SEP'] == 'P0')]['VALORIZACION'].sum()
p0_ambar_clp = data[(data['ALERTA ANTIG.'] == 'ÁMBAR') & (data['POLITICA CJE AL 30SEP'] == 'P0')]['VALORIZACION'].sum()

if p0_rojo > 0:
    p0_eval = "RIESGO ALTO"
elif p0_ambar > 0:
    p0_eval = "REVISAR"
else:
    p0_eval = "OK"

# Reglas de errores de datos (NA)
na = data.isna().any(axis=1).sum()
na_clp = data[data.isna().any(axis=1)]['VALORIZACION'].sum()

if na > 0:
    na_eval = "CORREGIR"
else:
    na_eval = "OK"

# Conciliación
print(f"---- Conciliación GL vs. Inventario ----\n")
print(f"Estado de conciliación: {conciliacion}")
print(f"Valor CLP: ${saldo_gl_cta_000184 - round(gran_total_jul)} \n")

# Antiguedad
print(f"---- Antigüedad ----\n")
print(f"Evaluación: {antiguedad_eval}")
print(f"Cantidad de ROJO: {rojos} | Valor CLP de ROJO: ${round(rojos_clp)}")
print(f"Cantidad de ÁMBAR: {ambar} | Valor CLP de ÁMBAR: ${round(ambar_clp)}")
print(f"Total de CLP para Antigüedad: ${round(rojos_clp) + round(ambar_clp)} \n")

# Fechas futuras FULCOM
print(f"---- Fechas FULCOM Futuras ----\n")
print(f"Evaluación: {fechas_futuras_eval}")
print(f"Cantidad de Fechas: {fechas_futuras} | Valor CLP de Fechas Futuras: ${round(fechas_futuras_clp)} \n")

# Conversión
print(f"---- Conversiones ----\n")
print(f"Evaluación: {conversiones_eval}")
print(f"Cantidad de Conversiones > 1: {conversiones_mayores} | Valor CLP de Conversiones > 1: ${round(conversiones_mayores_clp)}")
print(f"Cantidad de Conversiones Nulas: {conversiones_nulas} | Valor CLP de Conversiones Nulas: ${round(conversiones_nulas_clp)}")
print(f"Total de CLP para Conversiones: ${round(conversiones_mayores_clp) + round(conversiones_nulas_clp)} \n")

# Valorización
print(f"---- Valorización (Precio ✕ Costo = Valorización) ----\n")
print(f"Evaluación: {valorizacion_eval}")
print(f"Cantidad de Valorizaciones Inconsistentes: {valorizacion_inconsistente} | Valor CLP de Valorizaciones Inconsistentes: ${round(valorizacion_clp)} \n")

# Duplicados
print(f"---- Códigos Duplicados ----\n")
print(f"Evaluación: {duplicados_eval}")
print(f"Cantidad de Códigos Duplicados: {codigos_duplicados} | Valor CLP de Códigos Duplicados: ${round(duplicados_clp)} \n")

# Política sin canje (P0) envejecido
print(f"---- Póliticas Sin Canje (P0) Clasificadas como Rojo o Ámbar ----\n")
print(f"Evaluación: {p0_eval}")
print(f"Cantidad de P0 en ROJO: {p0_rojo} | Valor CLP de P0 en ROJO: ${round(p0_rojo_clp)}")
print(f"Cantidad de P0 en ÁMBAR: {p0_ambar} | Valor CLP de P0 en ÁMBAR: ${round(p0_ambar_clp)}")
print(f"Total de CLP para Políticas Sin Canje (P0): ${round(p0_rojo_clp) + round(p0_ambar_clp)} \n")

# Errores de datos (NA)
print(f"---- Nulos (NA) en Cualquier Registro ----\n")
print(f"Evaluación: {na_eval}")
print(f"Cantidad de Filas con Nulos: {na} | Valor CLP de Filas con Nulos: ${round(na_clp)} \n")

---- Conciliación GL vs. Inventario ----

Estado de conciliación: CONCILIADO
Valor CLP: $0 

---- Antigüedad ----

Evaluación: NO ALINEADO
Cantidad de ROJO: 1906 | Valor CLP de ROJO: $177783844
Cantidad de ÁMBAR: 1032 | Valor CLP de ÁMBAR: $647613372
Total de CLP para Antigüedad: $825397216 

---- Fechas FULCOM Futuras ----

Evaluación: CRÍTICO
Cantidad de Fechas: 10624 | Valor CLP de Fechas Futuras: $114457048186 

---- Conversiones ----

Evaluación: REVISAR
Cantidad de Conversiones > 1: 107 | Valor CLP de Conversiones > 1: $720286473
Cantidad de Conversiones Nulas: 2 | Valor CLP de Conversiones Nulas: $26000
Total de CLP para Conversiones: $720312473 

---- Valorización (Precio ✕ Costo = Valorización) ----

Evaluación: OK
Cantidad de Valorizaciones Inconsistentes: 0 | Valor CLP de Valorizaciones Inconsistentes: $0 

---- Códigos Duplicados ----

Evaluación: OK
Cantidad de Códigos Duplicados: 0 | Valor CLP de Códigos Duplicados: $0 

---- Póliticas Sin Canje (P0) Clasificadas como Roj

## Información del Dashboard

In [7]:
# Encabezado
print(f"---- Conciliación ----\n")
print(f"Saldo GL CTA_000184: ${saldo_gl_cta_000184} - Balance de Comprobación")
print(f"Total KARDEX: ${round(gran_total_jul)} - Total Existencias")
print(f"Diferencia KARDEX / GL: ${round(saldo_gl_cta_000184 - gran_total_jul)} (Debe estar dentro de tolerancia)")
print(f"Estado de Conciliación: {conciliacion} (Umbral de 0.1%) del GL")
print(f"SKU Únicos: {data['CODIGO'].unique().shape[0]} - Códigos Distintos")

---- Conciliación ----

Saldo GL CTA_000184: $117495620727 - Balance de Comprobación
Total KARDEX: $117495620727 - Total Existencias
Diferencia KARDEX / GL: $0 (Debe estar dentro de tolerancia)
Estado de Conciliación: CONCILIADO (Umbral de 0.1%) del GL
SKU Únicos: 15357 - Códigos Distintos


In [11]:
# Antiguedad de inventario
verdes = (data['ALERTA ANTIG.'] == 'VERDE').sum()
verdes_clp = data[data['ALERTA ANTIG.'] == 'VERDE']['VALORIZACION'].sum()

antiguedad_inventario =pd.DataFrame({
    'Clasificación': ['ROJO (> 12 meses)', 'ÁMBAR (7 - 12 meses)', 'VERDE (≤ 6 meses)'],
    'Partidas': [rojos, ambar, verdes],
    'Valoración CLP ($)': [round(rojos_clp), round(ambar_clp), round(verdes_clp)]
})

antiguedad_inventario

,Clasificación,Partidas,Valoración CLP ($)
0,ROJO (> 12 meses),1906,177783844
1,ÁMBAR (7 - 12 meses),1032,647613372
2,VERDE (≤ 6 meses),12419,116149721455


In [ ]:
# Top 10 ROJOS por VALORIZACIÓN 
top_10 = data[data['ALERTA ANTIG.'] == 'ROJO'][['CODIGO', 'DESCRIPTOR', 'MESES ANTIG.', 'VALORIZACION']]
top_10.sort_values(by = 'VALORIZACION', ascending = False, inplace = True)

top_10.head(10)


,CODIGO,DESCRIPTOR,MESES ANTIG.,VALORIZACION
5746,SKU_006155,PRODUCTO_006155,14.0,9570562.20
8359,SKU_009098,PRODUCTO_009097,29.0,7894440.84
6020,SKU_006458,PRODUCTO_006458,13.0,7161337.50
4933,SKU_005252,PRODUCTO_005252,38.0,6352395.22
5282,SKU_005637,PRODUCTO_005637,21.0,4157195.82
7819,SKU_008472,PRODUCTO_008472,13.0,3765563.76
6298,SKU_006766,PRODUCTO_006766,13.0,3479561.19
7262,SKU_007859,PRODUCTO_007859,16.0,3313694.08
9183,SKU_010003,PRODUCTO_010002,16.0,2697620.97
4150,SKU_004386,PRODUCTO_004386,29.0,2686446.40
